# Practical Lab: A Customer Feedback Chatbot (NLTK only)

In this lab you build a small chatbot for a fictional company, **BrightBox** (a weekly food box delivery service). Everything is done with **NLTK**. The bot reads customer messages and an FAQ, then:

1. **Scores sentiment** (positive, neutral, negative) for every message
2. **Decides what type of message it is**: complaint, question, request, praise or general comment
3. **Tags** every message: topics (delivery, billing and so on), entities (people, places, products, competitors, money, dates, order numbers) and an urgency flag
4. **Writes an automated response** that depends on the message type, the topic and the sentiment, and decides whether it is safe to send automatically or needs a person to approve it first
5. **Answers questions** about the feedback ("show negative delivery messages", "which cities are mentioned?")
6. **Answers FAQs** with TF-IDF search ("how do I pause my subscription?")
7. **Accepts live input**: type or paste any customer message and get its classification and a drafted reply

Everything runs **offline** once the libraries are installed. There are no API keys and no cloud calls, so it works the same in Google Colab and in VS Code.

### How this connects to the course

| Course part | Used here for |
|---|---|
| Part 1: regex, tokenising | Order numbers, money, dates, urgency words |
| Part 2: tagging, lemmas, stopwords | Matching "delivered", "delivering" and "delivery" to one topic |
| Part 3: TF-IDF, similarity, NER | FAQ search, similar messages, finding people |
| Part 4: sentiment, confusion matrix | Scoring messages and measuring how good the scoring and the message typing are |

### What the finished bot is made of

| Piece | NLTK tool | What it does |
|---|---|---|
| Sentiment | VADER, with our own domain words added | A positive or negative score per message |
| Message type | Rules on top of sentiment and `sent_tokenize` | Complaint, question, request, praise or comment |
| Topic tags | `word_tokenize`, `pos_tag`, WordNet lemmatiser | Which subjects a message is about |
| Entity tags | `ne_chunk` for people, plus a **gazetteer** and regex for our own names, money and dates | Who, where, how much, which product |
| Search | `FreqDist`, TF-IDF written with NLTK pieces, `cosine_distance` | FAQ answers and similar messages |
| Automated reply | Templates chosen by type, topic and sentiment | A drafted response and a send or review decision |
| Conversation | Plain Python rules | Decides what the user is asking for |

This is a **rule-based, classic NLP** bot. It is fast, free, transparent and easy to explain to a client. The last section covers where a large language model would improve it.

**Suggested timing:** about 80 minutes as a lab after Part 4, or split into two sessions (sections 1 to 4, then 5 to 8).

## 0. Setup

Run this cell once. It installs the libraries into the **same Python environment your notebook is using** (this is why we use `%pip` and not `!pip`, which can install into a different Python in VS Code) and downloads the NLTK data.

Only NLTK processes language here. pandas and matplotlib hold the results and draw the charts.

If the cell asks you to restart the kernel after installing, restart it and run the cell again.

In [ ]:
# 0. SETUP: installs go into the notebook's own environment (works in Colab and VS Code)
%pip install -q nltk pandas matplotlib numpy

import re
import math
import textwrap
import warnings
from collections import Counter
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
            "maxent_ne_chunker", "maxent_ne_chunker_tab", "words", "vader_lexicon"]:
    nltk.download(pkg, quiet=True)

from nltk import FreqDist, ngrams, word_tokenize, sent_tokenize, pos_tag, ne_chunk
from nltk.tree import Tree
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.cluster.util import cosine_distance
from nltk.metrics import precision, recall, f_measure

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 95)
pd.set_option("display.width", 150)

print("Ready. NLTK version:", nltk.__version__)

## 1. The information the bot will read

Two small data sets, both written for this lab:

- **`msgs`**: 40 customer messages from four channels (email, chat, app review, social). Each one has two **hand-written labels** that we treat as the right answer: the overall tone (`pos`, `neu`, `neg`) and the kind of message (`complaint`, `question`, `request`, `praise`, `comment`). Having labels lets us **measure** the sentiment scoring and the message typing instead of just trusting them
- **`faq`**: 10 question and answer pairs the bot can quote back

Some messages are deliberately tricky: mixed opinions ("great app but slow website"), polite requests with no emotion, and complaints written in a calm tone.

Real projects load this from a CSV or a database. See the note in section 8 on swapping in your own data.

In [ ]:
# 1. THE DATA: (channel, hand label, text). The label is what a person judged the overall tone to be.
raw_messages = [
    ("email",      "neg", "My Family Box (order BB-48213) arrived two days late and the vegetables were wilted. Really disappointed, Sarah Jones from Leeds."),
    ("app review", "pos", "Love the new app update! Checkout is so quick now and I can skip a week in two taps."),
    ("chat",       "neu", "Hi, can you tell me when my order BB-48370 will arrive? Tracking has not updated since Monday."),
    ("social",     "pos", "Absolutely delicious recipes this week. The Veggie Box is great value at £24.99, well done BrightBox!"),
    ("email",      "neg", "I was charged £59.98 twice for my subscription on 3 October. I need a refund immediately, this is unacceptable."),
    ("chat",       "pos", "Thanks so much to Priya on the support team, she sorted my refund in five minutes. Brilliant service."),
    ("app review", "neg", "The app keeps crashing when I try to log in. Tried reinstalling twice. Useless."),
    ("email",      "neu", "Could you confirm how to pause my subscription for two weeks in December? Thank you."),
    ("social",     "pos", "Switched from GreenPlate to BrightBox last month and the portions are much bigger. Happy customer in Bristol."),
    ("chat",       "neg", "The driver left my box outside in the rain and the packaging was soaked. Nothing was fresh."),
    ("email",      "neu", "Delivery was on time and the courier was friendly. The food was fine, nothing special."),
    ("app review", "neu", "Great app but the website is slow and I could not apply my discount code at checkout."),
    ("chat",       "neg", "I have been waiting 40 minutes on your helpline and nobody has answered. Awful customer service."),
    ("email",      "neg", "Please cancel my Student Box from next Friday. It is too expensive for me at the moment."),
    ("social",     "pos", "BrightBox Family Box is brilliant for busy parents. Kids actually eat the veg now!"),
    ("email",      "neg", "The salmon in my box smelled off and I threw it away. I want a refund for order BB-49002."),
    ("app review", "pos", "Easy to use, clear recipes and the delivery slots are flexible. Five stars."),
    ("chat",       "neu", "Hello, I moved to Manchester on 1 November. How do I update my delivery address?"),
    ("email",      "neg", "Your customer service team were rude to my mother when she rang about a missed delivery. I will be contacting Trading Standards."),
    ("social",     "neg", "Not impressed. Third late delivery this month. Going back to GreenPlate."),
    ("app review", "neu", "Does what it says. Nothing more, nothing less."),
    ("chat",       "pos", "Thank you! The replacement box arrived this morning and everything was perfect."),
    ("email",      "neg", "I love the variety of recipes but the price went up from £24.99 to £29.99 without any warning. Not happy about that."),
    ("app review", "neg", "The update broke the app on my phone. I cannot see my upcoming deliveries any more. Please fix this ASAP!"),
    ("social",     "neu", "Anyone tried the new vegan range from BrightBox? Looks interesting."),
    ("email",      "pos", "Wonderful experience from start to finish. Customer support in London answered within minutes and the box was fresh."),
    ("chat",       "neg", "My order BB-50111 is missing the chicken thighs. What can you do about that?"),
    ("email",      "neu", "Can I change the delivery day from Wednesday to Thursday?"),
    ("social",     "neg", "Meh. The recipes this week were bland and the portions tiny. Would not recommend."),
    ("app review", "pos", "Fantastic! The app now shows exactly when the driver will arrive. Saves me waiting in all day."),
    ("email",      "neg", "I am extremely angry. My card was charged £34.50 for a box I cancelled on 14 October. Fix this today!!!"),
    ("chat",       "neu", "Is the Veggie Box suitable for vegans?"),
    ("social",     "pos", "Good value for money and the ingredients taste really fresh. Will order again."),
    ("email",      "neg", "The courier delivered to the wrong address in Bristol and the neighbour has now thrown the food away. Terrible."),
    ("app review", "neg", "Password reset email never arrives. Been trying for an hour."),
    ("chat",       "pos", "Great chat with Tom, he explained the vegan range really well and even recommended a recipe."),
    ("email",      "neu", "Just letting you know the box arrived on Saturday as expected."),
    ("social",     "pos", "The Student Box is a lifesaver at £12 a week. Cheap, cheerful and tasty."),
    ("email",      "neg", "Still no refund after three weeks and four emails. Poor communication and no apology."),
    ("app review", "neu", "The new recipe filters are really helpful, but the app is slow to load on my old phone."),
]
msgs = pd.DataFrame(raw_messages, columns=["channel", "label", "text"])
msgs.insert(0, "id", range(1, len(msgs) + 1))

# A second hand label: what KIND of message is it? (used in section 3.4 to measure the message type rules)
msgs["hand_type"] = [
    "complaint", "praise", "question", "praise", "complaint", "praise", "complaint", "question", "praise", "complaint",
    "comment", "comment", "complaint", "request", "praise", "complaint", "praise", "question", "complaint", "complaint",
    "comment", "praise", "complaint", "complaint", "question", "praise", "complaint", "question", "complaint", "praise",
    "complaint", "question", "praise", "complaint", "complaint", "praise", "comment", "praise", "complaint", "comment",
]

faq = pd.DataFrame([
    ("How do I pause my subscription?",
     "Open the app, go to Subscription, then Pause. You can pause for 1 to 8 weeks, as long as you do it before the Friday cut-off."),
    ("How do I update my delivery address?",
     "Go to Account, then Delivery details, and enter the new address. Changes must be made before the Friday cut-off to apply to your next box."),
    ("Can I change my delivery day?",
     "Yes. Under Delivery details choose a new day. We deliver Monday to Saturday, and changes apply from your next box."),
    ("How do I get a refund or my money back?",
     "If something was wrong with your box or you were charged twice, report it in the app within 48 hours with a photo or screenshot. Approved refunds go back to your original payment method within 5 working days."),
    ("What should I do if an item is missing or damaged?",
     "Report it in the app within 48 hours with a photo. We will send a replacement or refund the item."),
    ("How do I cancel my subscription?",
     "Go to Subscription, then Cancel. Cancel before the Friday cut-off to avoid being charged for the next box."),
    ("Are your boxes suitable for vegans?",
     "The Veggie Box and the vegan range are fully vegan. The Family Box and Student Box contain meat and fish options."),
    ("I cannot log in or my password reset email has not arrived.",
     "Check your spam folder, then request a new reset link. If it still does not arrive, contact support in the app chat and we will reset it for you."),
    ("How can I track my delivery?",
     "Live tracking is in the app on delivery day, and we send a text message with a delivery window that morning."),
    ("How do I contact customer support?",
     "Use the chat in the app, open every day from 8am to 8pm, or email the support team. We reply to emails within one working day."),
], columns=["question", "answer"])

print(len(msgs), "messages |", len(faq), "FAQ entries")
print(msgs["channel"].value_counts().to_dict())
print(msgs["label"].value_counts().to_dict())
msgs.head(8)

## 2. Sentiment analysis

**Sentiment analysis** estimates the emotional tone of a text. We use **VADER**, which is built into NLTK. It is a word list made for **short, informal text**, plus rules for negation (`not good`), intensifiers (`very good`), capitals (`GOOD`) and exclamation marks.

It returns four numbers: `neg`, `neu` and `pos` (shares of the text) and a **compound** score from -1 (very negative) to +1 (very positive), which is the single overall number we use.

We turn the compound score into a label with a threshold: above the **band** is positive, below minus the band is negative, anything in between is neutral.

In [ ]:
# 2.1 HOW VADER SCORES A FEW SENTENCES
sia = SentimentIntensityAnalyzer()

for s in ["The delivery was good.",
          "The delivery was very good.",
          "The delivery was VERY GOOD!!!",
          "The delivery was not good.",
          "The delivery was late."]:
    print(f"{sia.polarity_scores(s)['compound']:+.2f}  {s}")
# Notice how intensifiers, capitals, exclamation marks and 'not' all change the score.
# Also notice 'late' scores 0: VADER does not know it is bad news in a delivery business.

In [ ]:
# 2.2 SCORE EVERY MESSAGE AND COMPARE WITH THE HAND LABELS
def to_label(score, band=0.05):
    """Turn a -1..+1 score into 'pos', 'neu' or 'neg'."""
    if score >= band:
        return "pos"
    if score <= -band:
        return "neg"
    return "neu"

msgs["vader"] = msgs["text"].apply(lambda t: sia.polarity_scores(t)["compound"])
msgs["vader_label"] = msgs["vader"].apply(to_label)
print(f"VADER agrees with the hand labels on {(msgs['vader_label'] == msgs['label']).mean():.0%} of messages")

msgs[["id", "label", "vader_label", "vader", "text"]].head(6)

### Reading the confusion matrix

This is the same idea as Part 4: rows are what the message **really** was (our hand label), columns are what the tool **predicted**. The diagonal is correct. Off the diagonal are the mistakes, and the pattern of mistakes tells you what to fix.

Questions to ask the room:

- Which kind of mistake is most common?
- Which is worse for a business: calling an angry customer neutral, or calling a neutral message negative?

In [ ]:
# 2.3 CONFUSION MATRIX, PRECISION AND RECALL FOR VADER (all from NLTK)
def report(gold, pred, labels=("neg", "neu", "pos")):
    """Print NLTK's confusion matrix and per-class precision, recall and F-measure."""
    gold, pred = list(gold), list(pred)
    print(nltk.ConfusionMatrix(gold, pred).pretty_format(sort_by_count=True))
    for lab in labels:
        ref = {i for i, g in enumerate(gold) if g == lab}
        tst = {i for i, p in enumerate(pred) if p == lab}
        p, r, f = precision(ref, tst), recall(ref, tst), f_measure(ref, tst)
        fmt = lambda x: "n/a" if x is None else f"{x:.2f}"
        print(f"{lab}: precision {fmt(p)}  recall {fmt(r)}  F {fmt(f)}  (n={len(ref)})")

report(msgs["label"], msgs["vader_label"])

# Read the mistakes. This is where improvement comes from.
errors = msgs[msgs["vader_label"] != msgs["label"]]
print(f"\n{len(errors)} mistakes. The biggest misses:")
errors.assign(gap=(errors["vader"] - errors["label"].map({"neg": -1, "neu": 0, "pos": 1})).abs()) \
      .sort_values("gap", ascending=False)[["id", "label", "vader_label", "vader", "text"]].head(8)

### Teaching the tool about our domain

Out of the box, VADER does not know that in a delivery business **late**, **missing**, **wilted**, **soaked** or **crashing** are bad news. The lexicon is a plain Python dictionary, so we can add those words. This is called **domain adaptation**, and it is often the cheapest way to improve a lexicon tool.

Two honest cautions:

1. We only have 40 messages, and we wrote the word list after reading them. The new score is therefore **optimistic**. In a real project you would check the improvement on **fresh labelled messages** the word list has never seen
2. We also tune the **neutral band**. A wider band calls more messages neutral, which helps with polite but mildly negative text

In [ ]:
# 2.4 DOMAIN ADAPTATION: ADD OUR OWN WORDS AND SCORES TO VADER (scale is -4 to +4)
DOMAIN_LEXICON = {
    "late": -1.8, "missing": -1.8, "wilted": -2.2, "soaked": -1.8, "wrong": -1.5,
    "crashing": -2.0, "crash": -2.0, "broke": -1.8, "slow": -1.3, "bland": -1.8,
    "tiny": -1.2, "waiting": -1.0, "expensive": -1.2, "unacceptable": -2.5,
    "refund": -0.8, "smelled": -1.0, "off": -0.6, "thrown": -1.0, "bigger": 0.8,
    "fresh": 1.5, "flexible": 1.2, "lifesaver": 2.5, "cheerful": 1.5,
}

sia_tuned = SentimentIntensityAnalyzer()
sia_tuned.lexicon.update(DOMAIN_LEXICON)

msgs["vader_tuned"] = msgs["text"].apply(lambda t: sia_tuned.polarity_scores(t)["compound"])

# Try several neutral bands for the original and the tuned analyser
rows = []
for band in [0.05, 0.10, 0.20, 0.30, 0.40]:
    for name, col in [("original", "vader"), ("tuned", "vader_tuned")]:
        pred = msgs[col].apply(lambda s: to_label(s, band))
        rows.append((name, band, round((pred == msgs["label"]).mean(), 3)))
sweep = pd.DataFrame(rows, columns=["lexicon", "band", "accuracy"]).pivot(index="band", columns="lexicon", values="accuracy")
print(sweep)

best_band = float(sweep["tuned"].idxmax())
print(f"\nBest band for the tuned lexicon on these 40 messages: {best_band}")

sweep.plot(marker="o", figsize=(6, 3.5))
plt.ylabel("Accuracy vs hand labels")
plt.title("Effect of tuning and of the neutral band")
plt.ylim(0, 1)
plt.show()

In [ ]:
# 2.5 LOCK IN THE SENTIMENT FUNCTION THE BOT WILL USE
BAND = best_band            # the neutral band we just chose

def sentiment(text):
    """Return (label, score) using the tuned VADER analyser."""
    score = sia_tuned.polarity_scores(text)["compound"]
    return to_label(score, BAND), score

msgs["sent_score"] = msgs["vader_tuned"]
msgs["sent_label"] = msgs["sent_score"].apply(lambda s: to_label(s, BAND))
print(f"Final sentiment accuracy on the labelled messages: {(msgs['sent_label'] == msgs['label']).mean():.0%}")
report(msgs["label"], msgs["sent_label"])

## 3. Tagging the information

A **tag** is a label that makes a message easy to filter and count. We create four kinds.

### 3.1 Topic tags (from lemmas)

We list the words that signal each topic and match on **lemmas** (Part 2). The pipeline is `word_tokenize`, then `pos_tag`, then the WordNet lemmatiser with the right part of speech, so `deliver`, `delivered`, `delivering` and `delivery` all trigger the **delivery** tag. One message can have several tags.

### 3.2 Entity tags (NLTK plus our own rules)

NLTK's `ne_chunk` finds people reasonably well, but it knows nothing about **our** products and competitors, it has no idea about money or dates, and it can mislabel places. So we combine three things:

| Source | Finds | Why |
|---|---|---|
| `ne_chunk` | People | The one thing the chunker does reasonably well, with a filter for obvious mistakes |
| A **gazetteer** (fixed lists) | Our products, the competitor, the cities we deliver to, Trading Standards | Our own names, so the model cannot get them wrong |
| Regex (Part 1) | Order numbers like `BB-48213`, money like `£59.98`, dates like `3 October` | Fixed shapes are what regex is for |

### 3.3 Flags (urgency and escalation)

A negative message should go to a human if **any** of these is true. Each rule gives a plain English reason, so nobody has to ask "why did the system flag this?":

| Rule | Example |
|---|---|
| **Urgent or legal wording** | "unacceptable", "ASAP", "Trading Standards" |
| **Very strong negative tone** | A sentiment score of -0.8 or lower |
| **Churn risk** | Mentions cancelling, switching, or a competitor |
| **Repeat contact** | "still no refund", "third late delivery", "nobody has answered" |

### 3.4 Message type

The last tag answers "what does this person want?" We use five types. The rules are written in plain code, and we **measure** them against the hand labels from section 1:

| Type | What it looks like |
|---|---|
| **complaint** | Negative sentiment |
| **question** | Contains a question mark, or starts like a question (*how*, *can*, *is*) |
| **request** | Asks us to do something (*please*, *can I*, *I want to*) without strong anger |
| **praise** | Positive sentiment and no question |
| **comment** | Everything else: neutral statements and mixed opinions |

In [ ]:
# 3.1 TOPIC VOCABULARY AND LEMMAS WITH NLTK
TOPIC_VOCAB = {
    "delivery": {"deliver", "delivery", "courier", "driver", "late", "arrive", "parcel", "package",
                 "tracking", "track", "dispatch", "missed", "slot", "address", "doorstep"},
    "billing": {"charge", "bill", "invoice", "refund", "payment", "card", "price", "cost", "expensive",
                "cheap", "subscription", "discount", "cancel", "pay", "value", "overcharge", "billing", "money"},
    "product quality": {"fresh", "wilted", "smell", "taste", "tasty", "delicious", "bland", "portion",
                        "ingredient", "recipe", "salmon", "chicken", "vegetable", "veg", "food",
                        "packaging", "soaked", "stale", "missing"},
    "customer service": {"support", "service", "helpline", "agent", "staff", "team", "rude", "reply",
                         "apology", "communication", "chat", "answer", "sort"},
    "app and website": {"app", "website", "login", "log", "crash", "update", "checkout", "password",
                        "reset", "filter", "load", "reinstall", "tap", "phone"},
}

lemmatizer = WordNetLemmatizer()
STOP = set(stopwords.words("english"))

def wn_pos(tag):
    """Convert a Penn Treebank tag (from pos_tag) into the tag WordNet expects."""
    return {"J": wordnet.ADJ, "V": wordnet.VERB, "R": wordnet.ADV}.get(tag[0], wordnet.NOUN)

@lru_cache(maxsize=None)
def lemma_pairs(text):
    """Tokenise, tag, and lemmatise. Returns a tuple of (word, lemma) for real words only."""
    return tuple((w.lower(), lemmatizer.lemmatize(w.lower(), wn_pos(tag)))
                 for w, tag in pos_tag(word_tokenize(text)) if w.isalpha())

def word_set(text):
    """Every word AND every lemma in the text, lowercased (so 'prices' and 'price' both match)."""
    pairs = lemma_pairs(text)
    return {w for w, _ in pairs} | {l for _, l in pairs}

def find_topics(text):
    """Return the sorted list of topics a text is about (empty list if none match)."""
    seen = word_set(text)
    return sorted(topic for topic, vocab in TOPIC_VOCAB.items() if seen & vocab)

def ranked_topics(text):
    """Matching topics, best match first (ties keep the order of TOPIC_VOCAB). Empty list if none match."""
    seen = word_set(text)
    scores = {topic: len(seen & vocab) for topic, vocab in TOPIC_VOCAB.items()}
    return [t for t in sorted(scores, key=lambda t: -scores[t]) if scores[t] > 0]

def primary_topic(text):
    """The topic with the most matching words (the one the reply should focus on)."""
    ranked = ranked_topics(text)
    return ranked[0] if ranked else "other"

# Try it
for s in ["The courier delivered my parcel late.", "Refunds are slow and the app keeps crashing."]:
    print(find_topics(s), "<-", s)
print(lemma_pairs("The couriers were delivering parcels")[:5])

In [ ]:
# 3.2 ENTITIES: NLTK ne_chunk FOR PEOPLE, A GAZETTEER AND REGEX FOR EVERYTHING ELSE

# First, what does the plain NLTK chunker make of a few sentences?
def chunk_entities(sentence):
    return [(" ".join(w for w, _ in n.leaves()), n.label())
            for n in ne_chunk(pos_tag(word_tokenize(sentence))) if isinstance(n, Tree)]

for s in ["Hello, I moved to Manchester on 1 November.",
          "Love the new app update!",
          "Switched from GreenPlate to BrightBox last month."]:
    print(chunk_entities(s), "<-", s)
# Plain ne_chunk mislabels our names and treats capitalised first words ('Love') as entities.

# Our gazetteer: fixed lists of names we know. These always win.
GAZETTEER = {
    "PRODUCT":    ["family box", "veggie box", "student box", "vegan range"],
    "COMPETITOR": ["greenplate"],
    "GPE":        ["manchester", "leeds", "bristol", "london"],
    "ORG":        ["trading standards"],
    "PERSON":     ["priya", "tom"],          # our own support staff, which the chunker often misses
}
GAZ_WORDS = {w for names in GAZETTEER.values() for n in names for w in n.split()} | {"brightbox"}

# Regex for things with a fixed shape (Part 1)
ORDER_RE = re.compile(r"\bBB-\d{5}\b")
MONEY_RE = re.compile(r"£\d[\d,]*(?:\.\d+)?")
MONTHS = "January|February|March|April|May|June|July|August|September|October|November|December"
DATE_RE = re.compile(rf"\b\d{{1,2}}(?:st|nd|rd|th)? (?:{MONTHS})\b")

def chunk_people(text):
    """People found by ne_chunk, with a filter for the commonest mistakes."""
    people = []
    for sentence in sent_tokenize(text):
        tree = ne_chunk(pos_tag(word_tokenize(sentence)))
        idx = 0                                           # position of the next token in the sentence
        for node in tree:
            if isinstance(node, Tree):
                tokens = [w for w, _ in node.leaves()]
                if node.label() == "PERSON" and all(t.isalpha() for t in tokens):
                    first_word_of_sentence = idx == 0
                    mistaken = any(t.lower() in GAZ_WORDS or t.lower() in STOP for t in tokens)
                    # A single capitalised word at the start of a sentence ('Love', 'Thanks') is rarely a name
                    if not mistaken and not (len(tokens) == 1 and first_word_of_sentence):
                        people.append(" ".join(tokens))
                idx += len(node.leaves())
            else:
                idx += 1
    return people

def find_entities(text):
    """Return a list of (text, label) for the entity types we care about."""
    out = [(name, "PERSON") for name in chunk_people(text)]
    for label, names in GAZETTEER.items():
        for name in names:
            for m in re.finditer(rf"\b{re.escape(name)}\b", text, re.IGNORECASE):
                out.append((m.group(0), label))
    out += [(m.group(0), "MONEY") for m in MONEY_RE.finditer(text)]
    out += [(m.group(0), "DATE") for m in DATE_RE.finditer(text)]
    return out

demo = "Sarah Jones in Leeds paid £59.98 for a Family Box (order BB-48213) on 3 October, then tried GreenPlate."
print("\nOur combined tagger:")
print(find_entities(demo), ORDER_RE.findall(demo))
print(find_entities("Hello, I moved to Manchester on 1 November."))

In [ ]:
# 3.3 FLAGS: urgency words, legal threats, and the escalation rule
URGENT_RE = re.compile(r"\b(urgent|urgently|immediately|asap|unacceptable|furious|angry|"
                       r"trading standards|solicitor|legal|ombudsman|fix this)\b", re.IGNORECASE)
CHURN_RE = re.compile(r"\b(cancel|cancelled|cancelling|switch|switched|switching|going back to|leaving)\b", re.IGNORECASE)
REPEAT_RE = re.compile(r"\b(still no|no (reply|response|answer)|nobody (has )?(answered|replied)|third|fourth|again)\b",
                       re.IGNORECASE)
STRONG_NEGATIVE = -0.8

def escalation(label, score, text, entities=()):
    """Return (escalate?, reason). Only negative messages are escalated, and every flag explains itself."""
    if label != "neg":
        return False, ""
    reasons = []
    if URGENT_RE.search(text):
        reasons.append("urgent or legal wording")
    if score <= STRONG_NEGATIVE:
        reasons.append("very strong negative tone")
    if CHURN_RE.search(text) or any(label_ == "COMPETITOR" for _, label_ in entities):
        reasons.append("churn risk")
    if REPEAT_RE.search(text):
        reasons.append("repeat contact")
    return (True, ", ".join(reasons)) if reasons else (False, "")

print(escalation("neg", -0.7, "This is unacceptable, I will contact Trading Standards."))
print(escalation("neg", -0.3, "The delivery was a bit late."))
print(escalation("neg", -0.1, "Third late delivery. Going back to GreenPlate.", [("GreenPlate", "COMPETITOR")]))

In [ ]:
# 3.4 MESSAGE TYPE: WHAT DOES THIS PERSON WANT?
QUESTION_START = re.compile(r"^(how|what|when|where|why|who|which|can|could|do|does|did|is|are|will|would|should|anyone|has|have)\b",
                            re.IGNORECASE)
REQUEST_RE = re.compile(r"\b(please|i (would like|want|need) to|can i|could you|cancel my|change my|update my|pause my)\b",
                        re.IGNORECASE)

def message_type(text, label, score):
    """Return one of: complaint, question, request, praise, comment."""
    first_sentence = sent_tokenize(text)[0].strip()
    asks = "?" in text or bool(QUESTION_START.match(first_sentence))
    if label == "pos" and not asks:
        return "praise"
    if label == "neg":
        # A calm 'please cancel...' is a request. Anything angrier is a complaint.
        if REQUEST_RE.match(first_sentence) and score > -0.5:
            return "request"
        return "complaint"
    if asks:
        return "question"
    if REQUEST_RE.search(text):
        return "request"
    return "comment"

msgs["msg_type"] = [message_type(t, l, s) for t, l, s in zip(msgs["text"], msgs["sent_label"], msgs["sent_score"])]
types = ["complaint", "question", "request", "praise", "comment"]
print(f"Message type agrees with the hand labels on {(msgs['msg_type'] == msgs['hand_type']).mean():.0%} of messages")
report(msgs["hand_type"], msgs["msg_type"], labels=types)

wrong = msgs[msgs["msg_type"] != msgs["hand_type"]]
print(f"\n{len(wrong)} disagreements:")
wrong[["id", "hand_type", "msg_type", "sent_label", "text"]]

In [ ]:
# 3.5 PUT IT ALL TOGETHER: ONE FUNCTION THAT TAGS ANY TEXT
def analyse(text):
    """Sentiment, type, topics, entities, order numbers and escalation for one text."""
    label, score = sentiment(text)
    entities = find_entities(text)
    escalate, reason = escalation(label, score, text, entities)
    return {
        "sent_label": label,
        "sent_score": round(score, 3),
        "msg_type": message_type(text, label, score),
        "topics": find_topics(text) or ["other"],
        "main_topic": primary_topic(text),
        "ranked": ranked_topics(text) or ["other"],
        "entities": entities,
        "orders": ORDER_RE.findall(text),
        "escalate": escalate,
        "reason": reason,
    }

tags = pd.DataFrame([analyse(t) for t in msgs["text"]])
msgs = pd.concat([msgs[["id", "channel", "label", "hand_type", "text"]], tags], axis=1)

print("Messages with no topic tag ('other'):", int(msgs["topics"].apply(lambda t: t == ["other"]).sum()))
print("Messages flagged for escalation     :", int(msgs["escalate"].sum()))
msgs[["id", "sent_label", "sent_score", "msg_type", "topics", "entities", "orders", "escalate"]].head(10)

### 3.6 What do the tags tell us?

Tags turn a pile of text into something you can count. Three views:

1. How many messages mention each topic
2. The **sentiment mix per topic**, which shows where the pain is
3. The most mentioned entities

In [ ]:
# 3.6 SUMMARISE THE TAGS
# One row per (message, topic) so a message with two topics counts for both
by_topic = msgs.explode("topics", ignore_index=True).rename(columns={"topics": "topic"})

# Chart 1: how often each topic is mentioned
counts = by_topic["topic"].value_counts()
counts.sort_values().plot.barh(figsize=(6.5, 3.2), color="#4c78a8")
plt.title("Messages per topic")
plt.xlabel("Number of messages")
plt.show()

# Chart 2: sentiment mix per topic (share of each topic's messages)
mix = pd.crosstab(by_topic["topic"], by_topic["sent_label"], normalize="index")
mix = mix.reindex(columns=["neg", "neu", "pos"], fill_value=0)
mix.loc[counts.index].plot.barh(stacked=True, figsize=(7, 3.6),
                                color={"neg": "#d1495b", "neu": "#b8b8b8", "pos": "#2a9d8f"})
plt.title("Sentiment mix by topic")
plt.xlabel("Share of messages")
plt.legend(title="sentiment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Table: the same numbers, with the average score
summary = by_topic.groupby("topic").agg(messages=("id", "count"),
                                        avg_sentiment=("sent_score", "mean"),
                                        pct_negative=("sent_label", lambda s: (s == "neg").mean()))
print(summary.sort_values("avg_sentiment").round(2))

# Most mentioned entities by type
ent_df = pd.DataFrame([(text, label) for ents in msgs["entities"] for text, label in ents], columns=["entity", "type"])
print()
for etype, group in ent_df.groupby("type"):
    print(f"{etype:<11}", dict(Counter(group["entity"]).most_common(5)))
print("ORDER IDs  ", [o for orders in msgs["orders"] for o in orders])

## 4. Search: answering FAQs and finding similar messages

The bot needs to answer "how do I pause my subscription?" without us writing a rule for every question. We use **TF-IDF and cosine similarity** (Part 3), written with NLTK building blocks:

1. Clean each FAQ entry and each message: lowercase **lemmas**, no stopwords (Part 2)
2. Turn them into TF-IDF vectors (smoothed, with unit length, as in section 3.2 of the course), including two-word phrases
3. Turn the user's question into a vector the same way
4. Return the entry with the highest `cosine_distance` similarity, but only if the score is high enough. A low score means "I do not know", and a bot that admits that is better than one that guesses

In [ ]:
# 4. TF-IDF SEARCH OVER THE FAQ AND OVER THE MESSAGES
FAQ_THRESHOLD = 0.15          # below this similarity, the bot says it does not know
AUTO_SEND_THRESHOLD = 0.35    # an FAQ answer is only sent automatically above this; weaker matches become drafts
DOMAIN_STOPWORDS = {"box"}    # appears in almost every entry in a box delivery business, so it tells us nothing
SYNONYMS = {"move": "change", "shift": "change", "rearrange": "change"}   # small fixes for words the FAQ spells differently

def prep(text):
    """Lowercase lemmas of the real words, with ordinary and domain stopwords removed."""
    return [SYNONYMS.get(lemma, lemma) for word, lemma in lemma_pairs(text)
            if word not in STOP and lemma not in STOP and lemma not in DOMAIN_STOPWORDS]

def cosine_sim(a, b):
    """Cosine similarity of two {word: weight} dictionaries, using NLTK's cosine_distance."""
    if not a or not b:
        return 0.0
    terms = sorted(set(a) | set(b))
    return 1 - cosine_distance([a.get(t, 0.0) for t in terms], [b.get(t, 0.0) for t in terms])

class TfidfIndex:
    """Smoothed TF-IDF over words and two-word phrases, searched with cosine similarity."""
    def __init__(self, token_docs):
        self.docs = [self.grams(d) for d in token_docs]
        n = len(self.docs)
        df = FreqDist()
        for d in self.docs:
            df.update(set(d))
        self.idf = {w: math.log((1 + n) / (1 + c)) + 1 for w, c in df.items()}
        self.vectors = [self.vector(d) for d in self.docs]

    @staticmethod
    def grams(tokens):
        return list(tokens) + [a + " " + b for a, b in ngrams(tokens, 2)]

    def vector(self, grams):
        counts = FreqDist(grams)
        v = {w: (1 + math.log(c)) * self.idf[w] for w, c in counts.items() if w in self.idf}   # sublinear tf
        length = math.sqrt(sum(x * x for x in v.values())) or 1.0
        return {w: x / length for w, x in v.items()}

    def search(self, text, top_n=1):
        q = self.vector(self.grams(prep(text)))
        sims = [cosine_sim(q, v) for v in self.vectors]
        order = sorted(range(len(sims)), key=lambda i: -sims[i])[:top_n]
        return [(i, float(sims[i])) for i in order]

# Index the FAQ on question + answer, so both can match
faq_index = TfidfIndex([prep(q + " " + a) for q, a in zip(faq["question"], faq["answer"])])
msg_index = TfidfIndex([prep(t) for t in msgs["text"]])

def search_faq(question, top_n=1):
    return faq_index.search(question, top_n)

def search_messages(query, top_n=3):
    return msg_index.search(query, top_n)

# Try some questions, including one the FAQ cannot answer
for q in ["How do I pause my subscription?",
          "my password reset email did not arrive",
          "Can I get my money back?",
          "I was charged twice, what do I do?",
          "I want compensation for the late box",
          "What is the capital of France?"]:
    i, score = search_faq(q)[0]
    answer = faq.loc[i, "question"] if score >= FAQ_THRESHOLD else "(no confident match, so the bot would say it does not know)"
    print(f"{score:.2f}  {q!r:<45} -> {answer}")
# TF-IDF matches WORDS, not meaning. 'money back' works because the FAQ entry uses those words.
# 'compensation' is not in the FAQ at all, so it cannot be matched. Embeddings fix that (see section 8).

## 5. Automated responses

Now the part that makes the bot useful in an inbox. For any new message the bot builds a reply from **three decisions**:

| Decision | Chosen by | Effect on the reply |
|---|---|---|
| **Message type** | Section 3.4 rules | The opening line: an apology for a complaint, thanks for praise, "thanks for getting in touch" for a question |
| **Topic** | The topic with the most matching words | The next step: what to do about a late box, a double charge, a crashing app |
| **Sentiment and escalation** | VADER score and the escalation rule | Whether a person is told to take over, and whether the reply is safe to send automatically |

A good automated reply follows a few rules, and the code enforces them:

1. **Never promise an outcome.** Say "we will check" and "approved refunds go back within 5 working days", not "you will get a refund"
2. **Quote the FAQ when it matches**, so the answer comes from your own policy and not from the bot's wording. A strong match (0.35 or more) can be sent automatically. A weak match (0.15 to 0.35) is still quoted, but only as a draft for a person to check, because TF-IDF matches words and not meaning
3. **Do not guess names.** A name in a message is often a member of staff or a third party, not the customer, so the bot never says "Hi Priya" just because Priya appears
4. **Separate "send" from "draft".** Low-risk replies (thanks for praise, a confident FAQ answer) can be sent automatically. Anything negative, escalated or uncertain is a **draft for a person to approve**

This is how you get the speed of automation without the risk of a bot saying the wrong thing to an angry customer.

In [ ]:
# 5. AUTOMATED REPLIES: TEMPLATES CHOSEN BY MESSAGE TYPE, TOPIC AND SENTIMENT
OPENING = {
    "complaint": "I am sorry about this, and thank you for telling us.",
    "question":  "Thanks for getting in touch.",
    "request":   "Thanks for letting us know what you need.",
    "praise":    "Thank you for the kind words, they really do make our day.",
    "comment":   "Thanks for the feedback.",
}

# What happens next, by topic, for complaints
COMPLAINT_NEXT = {
    "delivery": "If a box was late, missing or damaged, please report it in the app within 48 hours with a photo and we will arrange a replacement or refund.",
    "billing": "We will check the charges on your account. Where a refund is approved it goes back to your original payment method within 5 working days.",
    "product quality": "Please send us a photo of the item in the app within 48 hours and we will replace it or refund it.",
    "customer service": "I have passed your comments to the support team lead so they can look at what happened.",
    "app and website": "Please make sure you have the latest version of the app. If the problem continues, our app chat can sort out your account.",
    "other": "I have passed your message to the team so they can look into it.",
}

# For praise: who gets to hear about it
PRAISE_NEXT = {
    "delivery": "I will share this with our delivery team.",
    "billing": "I am glad the price and value feel right for you.",
    "product quality": "I will share this with our recipe and food teams.",
    "customer service": "I will share this with the support team.",
    "app and website": "I will share this with our app team.",
    "other": "I will share this with the team.",
}

COMMENT_NEXT = "I have added it to the feedback we share with the team each week."
NO_ANSWER = "I do not have an instant answer to that, so I have passed it to the team, who will reply within one working day."
ESCALATE_LINE = "Because of what you have described, I have asked a senior member of the team to contact you personally."
CLOSING = "If there is anything else we can do, just reply to this message.\n\nBrightBox Customer Care"

def build_reply(text, a):
    """Write a reply from the analysis dictionary. Returns (reply text, send mode)."""
    kind, topic = a["msg_type"], a["main_topic"]
    parts = [OPENING[kind]]
    faq_matched = False

    if a["orders"]:
        parts.append("I have noted order " + ", ".join(a["orders"]) + ".")

    if kind in ("question", "request"):
        (i, score), = search_faq(text)
        if score >= FAQ_THRESHOLD:
            parts.append(faq.loc[i, "answer"])
            faq_matched = score >= AUTO_SEND_THRESHOLD      # a weak match is shown, but a person must check it
        else:
            parts.append(NO_ANSWER)
    elif kind == "complaint":
        for t in a["ranked"][:2]:                           # the two best matching topics, best first
            parts.append(COMPLAINT_NEXT.get(t, COMPLAINT_NEXT["other"]))
    elif kind == "praise":
        staff = [name for name, label in a["entities"] if label == "PERSON"]
        if staff:
            parts.append(f"I will pass your thanks to {staff[0]}.")
        parts.append(PRAISE_NEXT.get(topic, PRAISE_NEXT["other"]))
    else:
        parts.append(COMMENT_NEXT)

    if a["escalate"]:
        parts.append(ESCALATE_LINE)
    parts.append(CLOSING)

    safe = (kind == "praise" or kind == "comment" or faq_matched) and not a["escalate"] and a["sent_label"] != "neg"
    return "\n\n".join(parts), ("send automatically" if safe else "draft for a person to approve")

def triage(text):
    """Everything the bot knows about a message, plus the drafted reply."""
    a = analyse(text)
    a["priority"] = "high" if a["escalate"] else ("normal" if a["sent_label"] == "neg" else "low")
    a["reply"], a["send_mode"] = build_reply(text, a)
    return a

def format_triage(a, show_reply=True):
    lines = [f"Type      : {a['msg_type']}",
             f"Sentiment : {a['sent_label']} ({a['sent_score']:+.2f})",
             f"Topics    : {', '.join(a['topics'])}  (main: {a['main_topic']})"]
    if a["entities"]:
        lines.append("Entities  : " + ", ".join(f"{t} [{l}]" for t, l in a["entities"]))
    if a["orders"]:
        lines.append("Orders    : " + ", ".join(a["orders"]))
    lines.append(f"Priority  : {a['priority']}" + (f"  (why: {a['reason']})" if a["escalate"] else ""))
    lines.append(f"Send mode : {a['send_mode']}")
    if show_reply:
        lines.append("\nDrafted reply:\n" + textwrap.indent(a["reply"], "  | "))
    return "\n".join(lines)

# Four very different messages, four very different replies (the first one gets two topic paragraphs)
for text in [
    "I was charged twice for order BB-60001 and nobody answers the helpline. This is unacceptable, I will contact Trading Standards.",
    "How do I pause my subscription for two weeks?",
    "Thank you Priya, the replacement Veggie Box arrived on Tuesday and it was lovely!",
    "The app keeps crashing when I open my deliveries.",
]:
    print("MESSAGE:", text)
    print(format_triage(triage(text)))
    print("\n" + "-" * 90 + "\n")

In [ ]:
# 5.1 REPLY TO ALL 40 MESSAGES AND REVIEW THE RESULTS
triaged = pd.DataFrame([triage(t) for t in msgs["text"]])
msgs["priority"] = triaged["priority"]
msgs["send_mode"] = triaged["send_mode"]
msgs["reply"] = triaged["reply"]

print("Message type by sentiment:")
print(pd.crosstab(msgs["msg_type"], msgs["sent_label"]))
print("\nWhat would happen to the 40 messages:")
print(msgs["send_mode"].value_counts().to_string())
print("\nPriority:", msgs["priority"].value_counts().to_dict())

# Chart: type and send mode together
pd.crosstab(msgs["msg_type"], msgs["send_mode"]).plot.barh(stacked=True, figsize=(7, 3.2),
                                                         color=["#e9a23b", "#2a9d8f"])
plt.title("Message types and what the bot would do")
plt.xlabel("Number of messages")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Read a few drafts against the original message. This is the check that matters.
for i in [0, 2, 27, 10]:
    row = msgs.iloc[i]
    print(f"[{row['id']}] {row['text']}")
    print(f"     -> {row['msg_type']}, {row['sent_label']}, {row['send_mode']}")
    print("     reply starts:", textwrap.shorten(row["reply"].replace("\n", " "), 150))
    print()

### Reading the drafts critically

Before you trust any automated reply, read a sample against the original messages. Questions to ask:

- Is the **type** right? A calm complaint can look like a question
- Does the reply **match the topic**? A late-box complaint should not get an app message
- Would you be comfortable if this went out **without** a person reading it? That is exactly what "send automatically" means, so keep that list short and boring
- Does any reply **promise** something the business has not agreed to?

## 6. The chatbot

The bot is a Python class with one public method, `reply(text)`. For each user message it decides **what the user wants** (the *intent*) using simple rules, then calls the pieces we already built.

| If the user says something like... | Intent | What the bot does |
|---|---|---|
| "hello", "help" | greeting, help | Introduces itself and lists examples |
| "give me an overview" | summary | Topic table: volume, average sentiment, percentage negative |
| "show negative delivery messages" | filter | Finds messages matching the topic, sentiment and channel it spotted |
| "how many billing complaints?" | count | Counts matching messages by sentiment |
| "which cities are mentioned?" | entities | Lists entities of that type, optionally filtered |
| "how do I pause my subscription?" | FAQ | Returns the best FAQ answer if it is similar enough |
| "analyse: <a new message>" | analyse | Type, sentiment, topics, entities, priority, and the drafted reply |
| "respond: <a new message>" | respond | Just the drafted reply and whether it can be sent automatically |
| "only the negative ones" | follow-up | Reuses the topic from the previous question |

**Notice that the bot uses the same `find_topics` and `sentiment` functions to understand the user's question as it did to tag the messages.** Tagging and understanding questions are the same job.

Everything is rule-based, so every answer can be explained. That matters in regulated industries, where you must be able to say why the system said what it said.

In [ ]:
# 6. THE CHATBOT
class FeedbackBot:
    NEG_WORDS = {"negative", "unhappy", "angry", "complaint", "complain", "bad", "upset", "problem", "issue", "poor"}
    POS_WORDS = {"positive", "happy", "praise", "compliment", "good", "great", "satisfied", "love"}
    CHANNELS = {"email": "email", "chat": "chat", "social": "social", "tweet": "social",
                "review": "app review"}
    ENTITY_WORDS = {"person": "PERSON", "people": "PERSON", "name": "PERSON", "who": "PERSON",
                    "city": "GPE", "cities": "GPE", "location": "GPE", "place": "GPE",
                    "competitor": "COMPETITOR", "rival": "COMPETITOR", "company": "COMPETITOR",
                    "amount": "MONEY", "spent": "MONEY", "money": "MONEY",
                    "date": "DATE",
                    "product": "PRODUCT"}
    FOLLOW_UP = {"those", "them", "these", "ones", "only", "just", "also"}
    LIST_WORDS = {"show", "list", "find", "customers", "messages", "feedback", "saying", "say", "complaints",
                  "reviews", "comments", "mention", "mentioned", "mentions"}

    def __init__(self, messages, faq_table):
        self.msgs = messages
        self.faq = faq_table
        self.last = {"topics": [], "sentiment": None, "channel": None}   # a little conversation memory

    # ---------- understanding the question ----------
    def parse(self, text):
        lemmas = word_set(text)
        topics = find_topics(text)
        sentiment_filter = "neg" if lemmas & self.NEG_WORDS else "pos" if lemmas & self.POS_WORDS else None
        channel = next((v for k, v in self.CHANNELS.items() if k in lemmas), None)
        entity = next((v for k, v in self.ENTITY_WORDS.items() if k in lemmas), None)
        if "order" in lemmas and {"number", "id", "reference"} & lemmas:
            entity = "ORDER"
        follow_up = bool(lemmas & self.FOLLOW_UP)
        return lemmas, topics, sentiment_filter, channel, entity, follow_up

    def filter_messages(self, topics, sentiment_filter, channel, urgent=False):
        m = self.msgs
        if topics:
            m = m[m["topics"].apply(lambda t: bool(set(t) & set(topics)))]
        if sentiment_filter:
            m = m[m["sent_label"] == sentiment_filter]
        if channel:
            m = m[m["channel"] == channel]
        if urgent:
            m = m[m["escalate"]]
        return m

    @staticmethod
    def describe(topics, sentiment_filter, channel, urgent=False):
        parts = []
        if topics:
            parts.append("topic: " + ", ".join(topics))
        if sentiment_filter:
            parts.append("sentiment: " + {"neg": "negative", "pos": "positive"}[sentiment_filter])
        if channel:
            parts.append("channel: " + channel)
        if urgent:
            parts.append("flagged for escalation")
        return "; ".join(parts) if parts else "all messages"

    @staticmethod
    def line(row, width=105):
        text = textwrap.shorten(row["text"], width=width, placeholder="...")
        return f"  [{row['id']:>2}] {row['channel']:<10} {row['sent_label']} ({row['sent_score']:+.2f})  {text}"

    # ---------- the different things the bot can do ----------
    def greet(self):
        return "Hello! I read BrightBox customer feedback and can draft replies. Type 'help' to see what I can do."

    def help(self):
        return textwrap.dedent("""\
            I can:
              - give an overview:        "give me an overview"
              - find messages:           "show negative delivery messages"
              - count things:            "how many billing complaints are there?"
              - follow up:               "only the ones from email"
              - list entities:           "which cities are mentioned?"  or  "which competitors are mentioned?"
              - answer FAQs:             "how do I pause my subscription?"
              - analyse a new message:   "analyse: my box was late and I want a refund"
              - draft a reply:           "respond: my box was late and I want a refund"
            Type 'quit' to leave.""")

    def summary(self):
        by_topic = self.msgs.explode("topics", ignore_index=True)
        table = by_topic.groupby("topics").agg(messages=("id", "count"),
                                               avg_sentiment=("sent_score", "mean"),
                                               pct_negative=("sent_label", lambda s: (s == "neg").mean() * 100))
        table = table.sort_values("avg_sentiment").round(2).rename_axis("topic")
        overall = self.msgs["sent_label"].value_counts().to_dict()
        types = self.msgs["msg_type"].value_counts().to_dict()
        esc = int(self.msgs["escalate"].sum())
        return (f"{len(self.msgs)} messages. Overall: {overall.get('pos', 0)} positive, "
                f"{overall.get('neu', 0)} neutral, {overall.get('neg', 0)} negative. {esc} flagged for escalation.\n"
                f"Types: {', '.join(f'{n} {k}' for k, n in types.items())}.\n"
                f"By topic (lowest sentiment first):\n{table.to_string()}")

    def show(self, topics, sentiment_filter, channel, urgent=False, limit=5):
        found = self.filter_messages(topics, sentiment_filter, channel, urgent)
        what = self.describe(topics, sentiment_filter, channel, urgent)
        if found.empty:
            return f"No messages match ({what})."
        ascending = sentiment_filter != "pos"
        found = found.sort_values("sent_score", ascending=ascending)
        lines = []
        for _, r in found.head(limit).iterrows():
            lines.append(self.line(r) + (f"\n        why: {r['reason']}" if urgent and r["reason"] else ""))
        more = f"\n  ...and {len(found) - limit} more." if len(found) > limit else ""
        return f"{len(found)} messages match ({what}). Showing the most {'positive' if not ascending else 'negative'}:\n" + "\n".join(lines) + more

    def count(self, topics, sentiment_filter, channel, urgent=False):
        found = self.filter_messages(topics, sentiment_filter, channel, urgent)
        what = self.describe(topics, sentiment_filter, channel, urgent)
        mix = found["sent_label"].value_counts().to_dict()
        return (f"{len(found)} messages ({what}): {mix.get('neg', 0)} negative, "
                f"{mix.get('neu', 0)} neutral, {mix.get('pos', 0)} positive.")

    def entities(self, label, topics, sentiment_filter, channel):
        found = self.filter_messages(topics, sentiment_filter, channel)
        what = self.describe(topics, sentiment_filter, channel)
        if label == "ORDER":
            items = [o for orders in found["orders"] for o in orders]
            name = "order numbers"
        else:
            items = [text for ents in found["entities"] for text, lab in ents if lab == label]
            name = {"PERSON": "people", "GPE": "places", "COMPETITOR": "competitors", "MONEY": "amounts of money",
                    "DATE": "dates", "PRODUCT": "products"}[label]
        if not items:
            return f"No {name} found ({what})."
        counted = ", ".join(f"{item} ({n})" for item, n in Counter(items).most_common(8))
        return f"{name.capitalize()} mentioned ({what}): {counted}"

    def answer_faq(self, text):
        (i, score), = search_faq(text)
        if score >= FAQ_THRESHOLD:
            return f"{self.faq.loc[i, 'answer']}\n  (FAQ: {self.faq.loc[i, 'question']}, match {score:.2f})"
        return None

    def analyse_new(self, text):
        return format_triage(triage(text))

    def respond_new(self, text):
        a = triage(text)
        return f"[{a['msg_type']} | {a['sent_label']} | {a['main_topic']} | {a['send_mode']}]\n" + a["reply"]

    # ---------- the single entry point ----------
    def reply(self, user_text):
        text = user_text.strip()
        low = text.lower()

        if re.match(r"^(hi|hello|hey|good (morning|afternoon|evening))\b", low) and len(low.split()) <= 4:
            return self.greet()
        if low in {"help", "?", "what can you do", "what can you do?"}:
            return self.help()

        m = re.match(r"^(respond|draft)\s*[:\-]?\s*(.+)$", text, re.IGNORECASE | re.DOTALL)
        if m:
            return self.respond_new(m.group(2).strip())
        m = re.match(r"^(analy[sz]e|tag|classify|score)\s*[:\-]?\s*(.+)$", text, re.IGNORECASE | re.DOTALL)
        if m:
            return self.analyse_new(m.group(2).strip())

        lemmas, topics, sent_f, channel, entity, follow_up = self.parse(text)

        # Follow-ups reuse what we were talking about, unless the user gave something new
        if follow_up and not (entity or {"overview", "summary"} & lemmas):
            topics = topics or self.last["topics"]
            channel = channel or self.last["channel"]
            sent_f = sent_f or self.last["sentiment"]

        urgent = bool({"urgent", "escalate", "escalation"} & lemmas)

        if {"overview", "summary", "breakdown", "report", "dashboard"} & lemmas:
            return self.summary()

        if entity:
            return self.entities(entity, topics, sent_f, channel)

        asks_count = bool(re.search(r"\bhow many\b|\bcount\b|\bnumber of\b", low))
        wants_messages = bool(self.LIST_WORDS & lemmas) or follow_up or urgent

        if asks_count and (topics or sent_f or channel or urgent or {"message", "feedback"} & lemmas):
            self.last = {"topics": topics, "sentiment": sent_f, "channel": channel}
            return self.count(topics, sent_f, channel, urgent)

        # A how-to question goes to the FAQ first, unless the user clearly asked to see messages
        if not wants_messages:
            faq_answer = self.answer_faq(text)
            if faq_answer:
                return faq_answer

        if topics or sent_f or channel or urgent:
            self.last = {"topics": topics, "sentiment": sent_f, "channel": channel}
            return self.show(topics, sent_f, channel, urgent)

        # Last resort: the closest messages, or an honest "I do not know"
        close = [(i, s) for i, s in search_messages(text) if s >= 0.15]
        if close:
            lines = [self.line(self.msgs.iloc[i]) for i, _ in close]
            return "I am not sure what you are asking, but these messages are the closest match:\n" + "\n".join(lines)
        return "Sorry, I did not understand that. Type 'help' for examples."

bot = FeedbackBot(msgs, faq)
print(bot.reply("help"))

## 7. Try the bot

First a scripted conversation, so you can see each intent working. Read the answers critically. The bot is not perfect, and the places where it slips are useful teaching moments.

In [ ]:
# 7.1 A SCRIPTED CONVERSATION
conversation = [
    "hello",
    "give me an overview",
    "show negative delivery messages",
    "only the ones from email",
    "how many billing complaints are there?",
    "what are customers saying about the app?",
    "only the negative ones",
    "which cities are mentioned?",
    "which competitors are mentioned in negative messages?",
    "what products do customers mention?",
    "show urgent messages",
    "how many urgent billing messages are there?",
    "how do I pause my subscription?",
    "I haven't received my password reset email",
    "analyse: I was charged twice for order BB-60001 and nobody answers the helpline. This is unacceptable, I will contact Trading Standards.",
    "respond: Thank you Priya, the replacement Veggie Box arrived on Tuesday and it was lovely!",
    "respond: Can I move my delivery to Thursday?",
    "what is the capital of France?",
]
for q in conversation:
    print("You:", q)
    print("Bot:", bot.reply(q))
    print()

### Things to notice and discuss

- **Follow-ups work** because the bot remembers the last topic, sentiment and channel. It is a tiny amount of state, and it already feels more like a conversation
- **The fallback is honest.** "What is the capital of France?" gets "Sorry, I did not understand", not a made-up answer
- **Every answer is traceable.** For any message you can see which topic words, entities and sentiment score produced it
- **The `analyse:` and `respond:` commands are the most commercially useful.** They are the start of an inbox triage tool: classify, score, draft a reply, and flag for a human

### Where it is weak (find these yourself)

1. Try `show negative messages about prices` (note `prices` versus `price`) and `show unhappy customers`. Which words does the bot understand and which does it ignore?
2. Try a mixed message with `respond:`, such as "great food but the delivery was late". Is one score and one type enough for a mixed message?
3. Look at message 27 ("My order BB-50111 is missing the chicken thighs. What can you do about that?"). Calm wording, but a real complaint. What did the tools say?
4. Ask `which people are mentioned?` Do you trust the list? Check it against the messages

In [ ]:
# 7.2 TRY YOUR OWN QUESTIONS HERE (edit the list and re-run)
my_questions = [
    "show negative messages about prices",
    "respond: great food but the delivery was late",
    "which people are mentioned?",
]
for q in my_questions:
    print("You:", q)
    print("Bot:", bot.reply(q))
    print()

## 8. Taking it further

### Use your own data

Replace the `msgs` data frame with a CSV that has at least a `text` column (and a `channel` column if you have one). Then re-run section 3.5 onwards.

```python
msgs = pd.read_csv("my_feedback.csv")          # needs columns: id, channel, text
msgs["label"] = ""                             # leave blank if you have no hand labels
msgs["hand_type"] = ""
```

Update four things for your own business: `TOPIC_VOCAB` (your topics and their words), the `GAZETTEER` (your products, competitors and places), `DOMAIN_LEXICON` (words that are good or bad news in your industry), and the reply templates in section 5 (your tone of voice and your real policies).

### Honest limits of this approach

| Limit | Why it happens | Fix |
|---|---|---|
| Keyword topics miss new wording | A topic only fires on words we listed | Train a topic classifier on labelled data (Part 4 techniques, NLTK's Naive Bayes), or use embeddings |
| One score and one type for a mixed message | Lexicon tools average everything | Score each sentence separately with `sent_tokenize`, then combine |
| Sarcasm and calm complaints | The words look neutral or positive | A trained model or a large language model |
| `ne_chunk` is not very accurate | It is a teaching-grade chunker | Keep the gazetteer for your own names, and check the people list by eye |
| Rules do not generalise | Every new question type needs a new rule | A language model to understand the question |
| Templates sound the same | Fixed wording | A language model to write the reply, with the facts and the send rules still in code |

### Upgrading the bot with a large language model

The structure we built is exactly the structure an LLM powered assistant would use:

- **The tags, scores and counts become tools** the model can call (a "search messages" tool, a "summary" tool)
- **The FAQ search becomes retrieval**: find the best passages with TF-IDF or embeddings, then give them to the model to write the answer from
- **The escalation rule and the send or draft decision stay in plain code**, so the decision about when to involve a person is always predictable and auditable

Keeping the facts, the tagging and the safety rules in code and using the model only for understanding questions and writing fluent answers is a good pattern for regulated work.

### Exercises

1. **Add a topic.** Create `"allergies and dietary"` with its own words, and a template in `COMPLAINT_NEXT` and `PRAISE_NEXT`. Check that the bot can filter by it and reply to it
2. **Add a competitor.** Add another name to the `GAZETTEER`, add a message that mentions it, and ask which competitors are mentioned
3. **Improve sentiment.** Add five of your own words to `DOMAIN_LEXICON`. Does accuracy go up, and is that still true on messages you did not use to choose the words?
4. **Improve the message type.** Look at the disagreements in section 3.4. Change one rule to fix one of them. Did another message break?
5. **Sentence level scoring.** Split each message into sentences with `sent_tokenize`, score each one, and report the most negative sentence. Does this handle "great food but the delivery was late" better?
6. **Safer sending.** Change the rules in `build_reply` so that nothing is ever sent automatically on a Friday afternoon. How would you test that?
7. **Measure the bot.** Write 15 test questions with the expected intent, run them through `bot.reply`, and calculate how many it gets right. A bot you cannot measure is a bot you cannot improve

## 9. Live input: classify a message and get a reply

Run the cell below, then type into the box that appears (in Colab it appears under the cell, in VS Code it appears at the top of the window). Choose a mode first:

| `LIVE_MODE` | What you type | What you get |
|---|---|---|
| `"inbox"` | Any customer message, such as "My box arrived late and half the food was warm" | The message type, sentiment, topics, entities, priority, whether it can be sent automatically, and a drafted reply |
| `"chat"` | A question about the feedback, such as "show negative billing messages" | The analyst chatbot from section 6 (it also understands `analyse:` and `respond:`) |
| `"off"` | Nothing | The cell does not wait for you, so "Run all" finishes |

Type `quit` to stop. In inbox mode, try the messages from the exercises: a calm complaint, a polite question, a thank you, an angry message with a legal threat, and a message that mixes praise and a problem.

In [ ]:
# 9. LIVE INPUT: choose a mode, then type. Type 'quit' to stop.
LIVE_MODE = "inbox"          # "inbox", "chat" or "off"

def inbox():
    print("INBOX MODE. Type or paste a customer message and press Enter. Type 'quit' to stop.")
    while True:
        try:
            text = input("\nCustomer message: ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if text.lower() in {"quit", "exit", "bye"}:
            print("Goodbye!")
            break
        if not text:
            continue
        print()
        print(format_triage(triage(text)))

def chat():
    print("Bot:", bot.reply("help"))
    while True:
        try:
            user = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if user.lower() in {"quit", "exit", "bye"}:
            print("Bot: Goodbye!")
            break
        if not user:
            continue
        print("Bot:", bot.reply(user))

if LIVE_MODE == "inbox":
    inbox()
elif LIVE_MODE == "chat":
    chat()